In [0]:
import mlflow
import pandas as pd
from pyspark.sql import functions as F
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, average_precision_score

CATALOG, SCHEMA, VOLUME = "workspace", "default", "opssat-ad"
T_GOLD = f"{CATALOG}.{SCHEMA}.gold_segment_features"
REFERENCE_FILE = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/dataset.csv"

PEAKS = ["n_peaks", "smooth20_n_peaks", "diff_peaks", "diff2_peaks"]
FEATURES = ["duration", "len", "mean", "var", "std", "kurtosis", "skew",
            "n_peaks", "smooth10_n_peaks", "smooth20_n_peaks", "diff_peaks", "diff2_peaks",
            "diff_var", "diff2_var", "gaps_squared", "len_weighted",
            "var_div_duration", "var_div_len"]

In [0]:
g = spark.table(T_GOLD)
ref = spark.read.option("header", True).option("inferSchema", True).csv(REFERENCE_FILE)

# Variante A : toutes mes features
mes_features = g

# Variante B : mes features, sauf les 4 pics remplacés par les officiels
officielles = g.drop(*PEAKS).join(ref.select("segment", *PEAKS), "segment")

pdf_a = mes_features.toPandas()
pdf_b = officielles.toPandas()
print(len(pdf_a), len(pdf_b))

In [0]:
mlflow.set_experiment(f"/Users/{spark.sql('select current_user()').first()[0]}/opssat-comparaison-pics")

def entrainer(pdf, nom):
    train, test = pdf[pdf["train"] == 1], pdf[pdf["train"] == 0]
    X_train, y_train = train[FEATURES], train["anomaly"]
    X_test, y_test = test[FEATURES], test["anomaly"]

    model = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42)
    model.fit(X_train, y_train)

    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)

    scores = {
        "f1": f1_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "auc_pr": average_precision_score(y_test, proba),
    }

    with mlflow.start_run(run_name=nom):
        mlflow.log_params({"variante": nom, "modele": "random_forest", "n_estimators": 300})
        mlflow.log_metrics(scores)

    importance = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
    return scores, importance

In [0]:
scores_a, imp_a = entrainer(pdf_a, "A_mes_pics")
scores_b, imp_b = entrainer(pdf_b, "B_pics_officiels")

display(pd.DataFrame({"A_mes_pics": scores_a, "B_pics_officiels": scores_b}).round(4).reset_index(names="metrique"))

display(pd.DataFrame({"A_mes_pics": imp_a, "B_pics_officiels": imp_b})
        .sort_values("A_mes_pics", ascending=False).round(4).reset_index(names="feature"))

In [0]:
SEEDS = [0, 1, 7, 42, 123]

def scores_seed(pdf, seed):
    train, test = pdf[pdf["train"] == 1], pdf[pdf["train"] == 0]
    model = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=seed)
    model.fit(train[FEATURES], train["anomaly"])
    proba = model.predict_proba(test[FEATURES])[:, 1]
    pred = (proba >= 0.5).astype(int)
    return {
        "f1": f1_score(test["anomaly"], pred),
        "recall": recall_score(test["anomaly"], pred),
        "auc_pr": average_precision_score(test["anomaly"], proba),
    }

lignes = []
for seed in SEEDS:
    for nom, pdf in [("A_mes_pics", pdf_a), ("B_pics_officiels", pdf_b)]:
        lignes.append({"variante": nom, "seed": seed, **scores_seed(pdf, seed)})

resultats = pd.DataFrame(lignes)
display(resultats)

In [0]:
synthese = resultats.groupby("variante")[["f1", "recall", "auc_pr"]].agg(["mean", "std"]).round(4)
display(synthese.reset_index())

ecart = resultats.pivot(index="seed", columns="variante", values="auc_pr")
ecart["B_moins_A"] = ecart["B_pics_officiels"] - ecart["A_mes_pics"]
display(ecart.reset_index())
print(f"B meilleur que A sur {(ecart['B_moins_A'] > 0).sum()} graines sur {len(SEEDS)}")

In [0]:
%pip install xgboost

In [0]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

def modeles(seed, pos_weight):
    return {
        "logistique": make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000)),
        "random_forest": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=seed),
        "xgboost": XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1,
                                 scale_pos_weight=pos_weight, random_state=seed, eval_metric="logloss"),
    }

train, test = pdf_b[pdf_b["train"] == 1], pdf_b[pdf_b["train"] == 0]
pos_weight = (train["anomaly"] == 0).sum() / (train["anomaly"] == 1).sum()

lignes = []
for seed in [0, 1, 7, 42, 123]:
    for nom, model in modeles(seed, pos_weight).items():
        model.fit(train[FEATURES], train["anomaly"])
        proba = model.predict_proba(test[FEATURES])[:, 1]
        pred = (proba >= 0.5).astype(int)
        scores = {"f1": f1_score(test["anomaly"], pred),
                  "recall": recall_score(test["anomaly"], pred),
                  "precision": precision_score(test["anomaly"], pred),
                  "auc_pr": average_precision_score(test["anomaly"], proba)}
        with mlflow.start_run(run_name=f"{nom}_seed{seed}"):
            mlflow.log_params({"modele": nom, "seed": seed, "features": "officielles"})
            mlflow.log_metrics(scores)
        lignes.append({"modele": nom, "seed": seed, **scores})

res = pd.DataFrame(lignes)
display(res.groupby("modele")[["f1", "recall", "precision", "auc_pr"]].agg(["mean", "std"]).round(4).reset_index())

In [0]:
import numpy as np
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_predict

mlflow.autolog(disable=True)

X_train, y_train = train[FEATURES], train["anomaly"]
X_test, y_test = test[FEATURES], test["anomaly"]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

espace = {
    "n_estimators": [200, 300, 500, 800],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5],
}

base = XGBClassifier(scale_pos_weight=pos_weight, eval_metric="logloss", random_state=42)
recherche = RandomizedSearchCV(base, espace, n_iter=40, scoring="average_precision",
                               cv=cv, random_state=42, n_jobs=-1)
recherche.fit(X_train, y_train)

print("Meilleurs paramètres :", recherche.best_params_)
print(f"AUC-PR en validation croisée : {recherche.best_score_:.4f}")

In [0]:
meilleur = recherche.best_estimator_
proba_oof = cross_val_predict(meilleur, X_train, y_train, cv=cv, method="predict_proba")[:, 1]

seuils = np.arange(0.10, 0.91, 0.05)
tableau = pd.DataFrame([{
    "seuil": round(s, 2),
    "f1": f1_score(y_train, proba_oof >= s),
    "recall": recall_score(y_train, proba_oof >= s),
    "precision": precision_score(y_train, proba_oof >= s),
} for s in seuils]).round(4)
display(tableau)

seuil_choisi = float(tableau.loc[tableau["f1"].idxmax(), "seuil"])
print(f"Seuil retenu (meilleur F1 en validation croisée) : {seuil_choisi}")

In [0]:
meilleur.fit(X_train, y_train)
proba_test = meilleur.predict_proba(X_test)[:, 1]

finaux = []
for nom, s in [("xgb_optimise_seuil_0.5", 0.5), (f"xgb_optimise_seuil_{seuil_choisi}", seuil_choisi)]:
    pred = (proba_test >= s).astype(int)
    scores = {"f1": f1_score(y_test, pred), "recall": recall_score(y_test, pred),
              "precision": precision_score(y_test, pred),
              "auc_pr": average_precision_score(y_test, proba_test)}
    with mlflow.start_run(run_name=nom):
        mlflow.log_params({**recherche.best_params_, "seuil": s, "features": "officielles"})
        mlflow.log_metrics(scores)
    finaux.append({"version": nom, **{k: round(v, 4) for k, v in scores.items()}})

display(pd.DataFrame(finaux))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

CATALOG, SCHEMA, VOLUME = "workspace", "default", "opssat-ad"
T_SILVER = f"{CATALOG}.{SCHEMA}.silver_telemetry"

In [0]:
T_EXTRA = f"{CATALOG}.{SCHEMA}.gold_features_extra"

w = Window.partitionBy("segment").orderBy("ts")

base = (
    spark.table(T_SILVER)
    .withColumn("t_sec", F.col("ts").cast("double"))
    .withColumn("prev", F.lag("value").over(w))
    .withColumn("diff", F.col("value") - F.col("prev"))
    .withColumn("gap", F.col("t_sec") - F.lag("t_sec").over(w))
    .withColumn("signe_change",
                (F.signum("diff") * F.signum(F.lag("diff").over(w)) < 0).cast("int"))
)

extra = (
    base.groupBy("segment")
    .agg(
        F.min("value").alias("min"),
        F.max("value").alias("max"),
        F.percentile_approx("value", 0.5).alias("mediane"),
        (F.percentile_approx("value", 0.75) - F.percentile_approx("value", 0.25)).alias("iqr"),
        F.max(F.abs("diff")).alias("saut_max"),
        F.avg(F.abs("diff")).alias("saut_moyen"),
        F.sum("signe_change").alias("changements_sens"),
        F.expr("regr_slope(value, t_sec)").alias("pente"),
        F.corr("value", "prev").alias("autocorr_1"),
        F.max("gap").alias("trou_max"),
    )
    .withColumn("amplitude", F.col("max") - F.col("min"))
    .fillna(0)
)

extra.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(T_EXTRA)
display(spark.table(T_EXTRA).limit(5))

In [0]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score

EXTRA = ["min", "max", "amplitude", "mediane", "iqr", "saut_max", "saut_moyen",
         "changements_sens", "pente", "autocorr_1", "trou_max"]

extra_pdf = spark.table("workspace.default.gold_features_extra").toPandas()
pdf_c = pdf_b.merge(extra_pdf, on="segment")
train_c = pdf_c[pdf_c["train"] == 1]

cv_rep = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
xgb = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1,
                    scale_pos_weight=pos_weight, eval_metric="logloss", random_state=42)

resultats_cv = []
for nom, cols in [("18_features", FEATURES), ("18_plus_extra", FEATURES + EXTRA)]:
    for metrique in ["f1", "average_precision"]:
        s = cross_val_score(xgb, train_c[cols], train_c["anomaly"], cv=cv_rep, scoring=metrique)
        resultats_cv.append({"jeu": nom, "metrique": metrique,
                             "moyenne": round(s.mean(), 4), "ecart_type": round(s.std(), 4)})

display(pd.DataFrame(resultats_cv))

In [0]:
f1_base = cross_val_score(xgb, train_c[FEATURES], train_c["anomaly"], cv=cv_rep, scoring="f1")
f1_extra = cross_val_score(xgb, train_c[FEATURES + EXTRA], train_c["anomaly"], cv=cv_rep, scoring="f1")
gain = f1_extra - f1_base
print(f"Les nouvelles features gagnent sur {(gain > 0).sum()} découpages sur {len(gain)}")
print(f"Gain moyen : {gain.mean():.4f} | écart-type du gain : {gain.std():.4f}")

xgb.fit(train_c[FEATURES + EXTRA], train_c["anomaly"])
importance = pd.Series(xgb.feature_importances_, index=FEATURES + EXTRA).sort_values(ascending=False)
display(importance.round(4).rename_axis("feature").reset_index(name="importance"))

In [0]:
test_c = pdf_c[pdf_c["train"] == 0]
xgb.fit(train_c[FEATURES + EXTRA], train_c["anomaly"])
proba = xgb.predict_proba(test_c[FEATURES + EXTRA])[:, 1]
pred = (proba >= 0.5).astype(int)

final = {"f1": f1_score(test_c["anomaly"], pred),
         "recall": recall_score(test_c["anomaly"], pred),
         "precision": precision_score(test_c["anomaly"], pred),
         "auc_pr": average_precision_score(test_c["anomaly"], proba)}

with mlflow.start_run(run_name="xgb_final_29_features"):
    mlflow.log_params({"features": "18_officielles_plus_11_extra", "seuil": 0.5})
    mlflow.log_metrics(final)

display(pd.DataFrame([final]).round(4))

In [0]:
from mlflow.models import infer_signature

mlflow.set_registry_uri("databricks-uc")
NOM_MODELE = "workspace.default.opssat_anomaly_xgb"

with mlflow.start_run(run_name="xgb_final_enregistre"):
    signature = infer_signature(train_c[FEATURES + EXTRA], xgb.predict_proba(train_c[FEATURES + EXTRA])[:, 1])
    mlflow.xgboost.log_model(
        xgb, artifact_path="model",
        signature=signature,
        input_example=train_c[FEATURES + EXTRA].head(3),
        registered_model_name=NOM_MODELE,
    )
    mlflow.log_metrics(final)

In [0]:
from mlflow import MlflowClient
MlflowClient().set_registered_model_alias(NOM_MODELE, "champion", 1)